# 🎯 Data Quality Validation - Simplified
## Shoebadoo Sales Analytics

---

## Was machen wir hier?

Wir validieren die **bereinigten Daten** mit **5 wichtigen Quality Checks**:

| # | Check | Warum wichtig? |
|---|-------|----------------|
| 1 | **Pflichtfelder** | Keine Sales ohne Customer/Product |
| 2 | **Wertebereich** | Quantity & Price müssen > 0 sein |
| 3 | **Primary Keys** | Keine doppelten IDs |
| 4 | **Datentypen** | Numerische Felder sind wirklich Zahlen |
| 5 | **Referenzen** | Alle product_ids existieren in Products |

---

## 1. Setup

In [1]:
import pandas as pd
import great_expectations as gx
from great_expectations.core.batch import RuntimeBatchRequest
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries geladen")
print(f"Great Expectations: {gx.__version__}")

✅ Libraries geladen
Great Expectations: 0.18.12


## 2. Daten laden

In [2]:
# Pfad zu den bereinigten Daten
INPUT_PATH = "/app/data/cleaned/2_data_cleaning"

# Daten laden (Pandas ist hier einfacher als PySpark)
sales_df = pd.read_parquet(f"{INPUT_PATH}/sales_clean.parquet")
products_df = pd.read_parquet(f"{INPUT_PATH}/products_clean.parquet")
customers_df = pd.read_parquet(f"{INPUT_PATH}/customers_clean.parquet")

print(f"✅ Sales: {len(sales_df):,} rows")
print(f"✅ Products: {len(products_df):,} rows")
print(f"✅ Customers: {len(customers_df):,} rows")

✅ Sales: 397,962 rows
✅ Products: 454 rows
✅ Customers: 8,000 rows


## 3. Great Expectations Setup (einfach)

In [3]:
# Great Expectations Context erstellen
context = gx.get_context()

# Datasource für unsere Pandas DataFrames
datasource = context.sources.add_or_update_pandas(name="shoebadoo_data")

# Data Assets registrieren
sales_asset = datasource.add_dataframe_asset(name="sales_clean")
products_asset = datasource.add_dataframe_asset(name="products_clean")

print("✅ Great Expectations Context erstellt")
print("✅ Datasource registriert")

✅ Great Expectations Context erstellt
✅ Datasource registriert


## 4. Quality Checks definieren

Jetzt kommt der wichtige Teil: Wir definieren unsere **5 Core Checks**.

In [4]:
# Expectation Suite erstellen
suite = context.add_or_update_expectation_suite("shoebadoo_quality_checks")

# Batch Request für Sales
batch_request = sales_asset.build_batch_request(dataframe=sales_df)
validator = context.get_validator(
    batch_request=batch_request,
    expectation_suite_name="shoebadoo_quality_checks"
)

print("✅ Validator erstellt für sales_clean")
print(f"📊 Validiere {len(sales_df)} Zeilen")

✅ Validator erstellt für sales_clean
📊 Validiere 397962 Zeilen


### 4.1 Check 1: Pflichtfelder (NOT NULL)

In [5]:
# Pflichtfelder dürfen nicht NULL sein
validator.expect_column_values_to_not_be_null(
    column="sale_id",
    meta={"severity": "CRITICAL", "description": "Jeder Sale braucht eine ID"}
)

validator.expect_column_values_to_not_be_null(
    column="customer_id",
    meta={"severity": "CRITICAL", "description": "Kein Sale ohne Customer"}
)

validator.expect_column_values_to_not_be_null(
    column="product_id",
    meta={"severity": "CRITICAL", "description": "Kein Sale ohne Product"}
)

print("✅ Check 1: Pflichtfelder definiert")

Calculating Metrics:   0%|          | 0/6 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/6 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/6 [00:00<?, ?it/s]

✅ Check 1: Pflichtfelder definiert


### 4.2 Check 2: Wertebereich (MIN/MAX)

In [6]:
# Quantity muss mindestens 1 sein
validator.expect_column_values_to_be_between(
    column="quantity",
    min_value=1,
    max_value=None,
    meta={"severity": "CRITICAL", "description": "Mindestens 1 Artikel verkauft"}
)

# Total Amount muss positiv sein
validator.expect_column_values_to_be_between(
    column="total_amount",
    min_value=0.01,
    max_value=None,
    meta={"severity": "CRITICAL", "description": "Preis muss > 0 sein"}
)

print("✅ Check 2: Wertebereiche definiert")

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

✅ Check 2: Wertebereiche definiert


### 4.3 Check 3: Primary Key (Eindeutigkeit)

In [7]:
# sale_id muss eindeutig sein
validator.expect_column_values_to_be_unique(
    column="sale_id",
    meta={"severity": "CRITICAL", "description": "Keine doppelten Sale IDs"}
)

print("✅ Check 3: Primary Key Uniqueness definiert")

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

✅ Check 3: Primary Key Uniqueness definiert


### 4.4 Check 4: Datentypen

In [8]:
# Quantity muss Integer sein
validator.expect_column_values_to_be_of_type(
    column="quantity",
    type_="int64",
    meta={"severity": "HIGH", "description": "Quantity ist eine Ganzzahl"}
)

# Total Amount muss numerisch sein
validator.expect_column_values_to_be_of_type(
    column="total_amount",
    type_="float64",
    meta={"severity": "HIGH", "description": "Betrag ist eine Dezimalzahl"}
)

print("✅ Check 4: Datentypen definiert")

Calculating Metrics:   0%|          | 0/1 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Check 4: Datentypen definiert


### 4.5 Check 5: Referentielle Integrität

In [9]:
# Alle product_ids in Sales müssen in Products existieren
valid_product_ids = products_df['product_id'].unique().tolist()

validator.expect_column_values_to_be_in_set(
    column="product_id",
    value_set=valid_product_ids,
    meta={"severity": "HIGH", "description": "Nur bekannte Products verkaufen"}
)

print("✅ Check 5: Referentielle Integrität definiert")
print(f"   Validiere gegen {len(valid_product_ids)} bekannte Products")

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

✅ Check 5: Referentielle Integrität definiert
   Validiere gegen 454 bekannte Products


## 5. Validation ausführen

In [10]:
print("="*80)
print("🔍 STARTE QUALITY VALIDATION")
print("="*80)
print()

# Suite speichern
validator.save_expectation_suite(discard_failed_expectations=False)

# Checkpoint erstellen und ausführen
checkpoint = context.add_or_update_checkpoint(
    name="shoebadoo_checkpoint",
    validations=[
        {
            "batch_request": batch_request,
            "expectation_suite_name": "shoebadoo_quality_checks"
        }
    ]
)

# VALIDATION DURCHFÜHREN
results = checkpoint.run()

print("="*80)
print("✅ VALIDATION ABGESCHLOSSEN")
print("="*80)

🔍 STARTE QUALITY VALIDATION



Calculating Metrics:   0%|          | 0/42 [00:00<?, ?it/s]

✅ VALIDATION ABGESCHLOSSEN


## 6. Ergebnisse anzeigen

In [11]:
# Ergebnisse auslesen
validation_result = results.list_validation_results()[0]

print("\n" + "="*80)
print("📊 QUALITY VALIDATION REPORT")
print("="*80)
print()

# Gesamtstatus
success = validation_result["success"]
status_icon = "✅" if success else "❌"
print(f"{status_icon} Gesamtstatus: {'PASSED' if success else 'FAILED'}")
print()

# Einzelne Checks
results_list = validation_result["results"]
print(f"📋 Anzahl Checks: {len(results_list)}")
print()

passed = sum(1 for r in results_list if r["success"])
failed = len(results_list) - passed

print(f"✅ Passed: {passed}")
print(f"❌ Failed: {failed}")
print()

# Details für failed checks
if failed > 0:
    print("\n⚠️  FAILED CHECKS:")
    print("-" * 80)
    for result in results_list:
        if not result["success"]:
            expectation = result["expectation_config"]["expectation_type"]
            column = result["expectation_config"].get("kwargs", {}).get("column", "N/A")
            print(f"❌ {expectation}")
            print(f"   Column: {column}")
            if "observed_value" in result["result"]:
                print(f"   Observed: {result['result']['observed_value']}")
            print()

print("="*80)


📊 QUALITY VALIDATION REPORT

✅ Gesamtstatus: PASSED

📋 Anzahl Checks: 9

✅ Passed: 9
❌ Failed: 0



## 8. HTML Report generieren (optional)

In [13]:
# Data Docs erstellen (HTML Report)
context.build_data_docs()

print("✅ HTML Report generiert")
print("📄 Location: /app/gx/uncommitted/data_docs/local_site/index.html")
print()
print("💡 Tipp: Öffne den Report im Browser für detaillierte Visualisierungen!")

✅ HTML Report generiert
📄 Location: /app/gx/uncommitted/data_docs/local_site/index.html

💡 Tipp: Öffne den Report im Browser für detaillierte Visualisierungen!


## 7️. Summary als DataFrame

In [12]:
# Erstelle übersichtliche Tabelle
summary_data = []

for idx, result in enumerate(results_list, 1):
    config = result["expectation_config"]
    expectation_type = config["expectation_type"]
    column = config.get("kwargs", {}).get("column", "N/A")
    success = "✅ PASS" if result["success"] else "❌ FAIL"
    
    # Severity aus Meta holen
    severity = config.get("meta", {}).get("severity", "MEDIUM")
    
    summary_data.append({
        "Check #": idx,
        "Column": column,
        "Expectation": expectation_type.replace("expect_column_", ""),
        "Result": success,
        "Severity": severity
    })

summary_df = pd.DataFrame(summary_data)
print("\n📊 CHECK SUMMARY")
print("="*80)
print(summary_df.to_string(index=False))
print("="*80)


📊 CHECK SUMMARY
 Check #       Column           Expectation Result Severity
       1      sale_id values_to_not_be_null ✅ PASS CRITICAL
       2      sale_id   values_to_be_unique ✅ PASS CRITICAL
       3  customer_id values_to_not_be_null ✅ PASS CRITICAL
       4   product_id values_to_not_be_null ✅ PASS CRITICAL
       5   product_id   values_to_be_in_set ✅ PASS     HIGH
       6     quantity  values_to_be_between ✅ PASS CRITICAL
       7     quantity  values_to_be_of_type ✅ PASS     HIGH
       8 total_amount  values_to_be_between ✅ PASS CRITICAL
       9 total_amount  values_to_be_of_type ✅ PASS     HIGH


---

## ✅ Fertig!

### Was haben wir erreicht?

1. ✅ **5 Core Quality Checks** implementiert
2. ✅ **Great Expectations** genutzt (Standard in der Industrie)
3. ✅ **Automated Validation** mit klarem Report
4. ✅ **HTML Documentation** generiert

### Nächste Schritte

- 📊 Star Schema Modeling (Kimball Methodology)
- 🚀 Dashboard mit Streamlit
- 📦 Docker Deployment

---